# Task 2.2 Mining Cancer Feature Patterns
Objective: 
- To analyse the `feature sequences` and `patterns` in cancer diagnosis data to uncover common charateristics that distinguish malignant from benign cases through sequential pattern mining.

Dataset Used: 
- Breast Cancer Dataset
- 569 patients with breast cancer
- 30 numerical features
- Target: Diagnosis (M = Malignant, B = Benign)

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.feature_selection import mutual_info_classif
from collections import defaultdict, Counter
from prefixspan import PrefixSpan
import time

## Task 2.2.1 Data Preprocessing

### 1. Data Preparation

In [2]:
# Load data
data = pd.read_csv("data/cancer-data.csv")

# Drop column with all NaN values
data = data.drop(columns=["Unnamed: 32"], errors='ignore')
data.info()

# Separate features and label
X = data.drop(columns=["id", "diagnosis"])
y = data["diagnosis"].map({"M": 1, "B": 0})
feature_names = X.columns

# Normalize data
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [ ]:
print(f"Dataset: {len(data)} patients, {len(feature_names)} features")
print(f"Malignant: {sum(y==1)}, Benign: {sum(y==0)}")

Dataset: 569 patients, 30 features
Malignant: 212, Benign: 357


### 2. Sequence Generation
To convert numerical cancer features into sequential patterns for pattern mining, we adopted **3 different ranking strategies**:

#### **a. Z-Score Ranking (make_sequence_zscore)**

Converts a patient's features into a sequence ordered by **magnitude of deviation from normal**:

1. **Rank features** by absolute z-score (how far from mean)
2. **Select top k** most extreme features
3. **Label each** as "high" (positive z-score) or "low" (negative z-score)
4. **Group into itemsets**: Features with similar z-scores (within `tie_threshold=0.01`, i.e., features within 0.01 standard deviations) are grouped together as simultaneous events
5. **Returns**: List of itemsets ordered by importance
   - Example: `[('radius_mean_high',), ('texture_mean_low', 'area_mean_high')]`

#### **b. Mutual Information Ranking (make_sequence_mi)**
Orders features by **global importance** for diagnosis prediction:

1. **Uses pre-computed** `mi_scores` (mutual information with diagnosis label)
2. **Selects top k** features by MI score (same order for all patients)
3. **For each feature**, labels as "high" or "low" relative to dataset mean
4. **Returns**: Simple list of strings
   - Example: `['radius_mean_high', 'texture_mean_low', ...]`


#### **c. Discretization-Based (make_sequence_binned)**
Bins continuous values into categorical levels:

1. **Discretizes** all features into `n_bins=3` categories using different strategies:
   - **"quantile"**: Equal-sized bins by percentile
   - **"uniform"**: Equal-width bins
   - **"kmeans"**: K-means clustering
2. **Labels bins** as 'low', 'med', 'high'
3. **Pairs items** into 2-element tuples for all patients
4. **Returns**: List of sequences (one per patient), each containing tuples
   - Example: `[(feat1_low, feat2_high), (feat3_med, feat4_low), ...]`

In [ ]:
# Sequence generation functions
def make_sequence_zscore(x_row, top_k=10, tie_threshold=0.01):
    """Create sequences with proper itemset grouping based on z-scores"""
    ranked = x_row.abs().sort_values(ascending=False)
    top_features = ranked.head(top_k).index.tolist()
    
    sequence = []
    current_itemset = []
    prev_zscore = None
    
    for feature in top_features:
        abs_zscore = abs(x_row[feature])
        label = "high" if x_row[feature] > 0 else "low"
        token = f"{feature}_{label}"
        
        if prev_zscore is not None and abs(abs_zscore - prev_zscore) <= tie_threshold:
            current_itemset.append(token)
        else:
            if current_itemset:
                sequence.append(tuple(sorted(current_itemset)))
            current_itemset = [token]
        prev_zscore = abs_zscore
    
    if current_itemset:
        sequence.append(tuple(sorted(current_itemset)))
    
    return sequence

def make_sequence_mi(x_row, mi_scores, X_scaled, top_k=10):
    """Order features by global mutual information importance"""
    top_features = mi_scores.head(top_k).index.tolist()
    sequence = []
    for feature in top_features:
        val = x_row[feature]
        mean_val = X_scaled[feature].mean()
        label = "high" if val > mean_val else "low"
        sequence.append(f"{feature}_{label}")
    return sequence

def make_sequence_binned(X, n_bins=3, strategy="quantile", top_k=10):
    """Binning-based sequence generation"""
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy=strategy)
    X_binned = pd.DataFrame(discretizer.fit_transform(X), columns=X.columns)
    
    sequences = []
    labels = ['low', 'med', 'high']
    
    for _, row in X_binned.iterrows():
        seq_items = [f"{feature}_{labels[int(row[feature])]}" 
                     for feature in X.columns[:top_k]]
        itemsets = [tuple(seq_items[i:i+2]) for i in range(0, len(seq_items), 2)]
        sequences.append(itemsets)
    
    return sequences

In [4]:
# Feature importance
mi_scores = pd.Series(
    mutual_info_classif(X_scaled, y, random_state=42),
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Features by Mutual Information:")
print(mi_scores.head(10))


Top 10 Features by Mutual Information:
perimeter_worst         0.471842
area_worst              0.464313
radius_worst            0.451230
concave points_mean     0.438806
concave points_worst    0.436255
perimeter_mean          0.402361
concavity_mean          0.375447
radius_mean             0.362276
area_mean               0.360023
area_se                 0.340759
dtype: float64


In [ ]:
# Generate sequences
# Semantic 1: Z-score ranking per patient
seqs_z = X_scaled.apply(make_sequence_zscore, axis=1).tolist()

# Semantic 2: Mutual Information–based ranking
mi_scores = pd.Series(mutual_info_classif(X_scaled, y), index=X.columns).sort_values(ascending=False)
seqs_mi = X_scaled.apply(lambda row: make_sequence_mi(row, mi_scores, X_scaled, top_k=5), axis=1)

# Semantic 3: Binning (low/med/high)
strategies = ["uniform", "quantile", "kmeans"]
seqs_binned_all = {}
for strat in strategies:
    seqs_binned_all[strat] = make_sequence_binned(X_scaled, n_bins=3, strategy=strat)

# Combine into a DataFrame
sequences_df = pd.DataFrame({
    "id": data["id"],
    "diagnosis": data["diagnosis"],
    "seq_zscore": seqs_z,
    "seq_mi": seqs_mi,
    "seq_binned_uniform": seqs_binned_all["uniform"],
    "seq_binned_quantile": seqs_binned_all["quantile"],
    "seq_binned_kmeans": seqs_binned_all["kmeans"],
})

# Save results
sequences_df.to_csv("data/sequence_semantics.csv", index=False)
print("Generated sequence representations saved to data/sequence_semantics.csv")

c:\Users\User\miniconda3\envs\venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


Generated sequences for 569 patients


## Task 2.2.2 Data Analysis

We find the frequently occurring ordered patterns in the generated sequence data using two sequential pattern mining algorithms:

### a. GSP (Generalized Sequential Pattern)
- **Strategy**: Breadth-first search (like Apriori algorithm)
- **Approach**: 
  - Generates candidate patterns level by level
  - Joins frequent k-sequences to create (k+1)-sequence candidates
  - Scans database multiple times to count support
  - Preserves itemset structure (simultaneous events)
- **Advantages**: 
  - Exhaustive pattern discovery
  - Intuitive candidate generation
- **Disadvantages**: 
  - Multiple database scans required
  - Large candidate set generation
  - Slower on large datasets

### b. PrefixSpan (Prefix-Projected Sequential Pattern Mining)
- **Strategy**: Depth-first search with database projection
- **Approach**:
  - Grows patterns recursively without candidate generation
  - Projects database into smaller subsets for each prefix
  - Explores only patterns that exist in the data
  - More efficient memory usage through projection
- **Advantages**:
  - Faster since no candidate generation overhead
  - Efficient pattern growth
- **Disadvantages**:
  - More complex implementation
  - In our implementation: flattens itemsets (loses simultaneous event structure)
  - Recursive overhead for very long patterns

In [ ]:
# GSP algorithm
def is_subsequence(pattern, sequence):
    """Check if pattern appears in sequence in order"""
    pat_idx = 0
    for itemset in sequence:
        if pat_idx >= len(pattern):
            return True
        pattern_set = set(pattern[pat_idx])
        sequence_set = set(itemset)
        if pattern_set.issubset(sequence_set):
            pat_idx += 1
    return pat_idx >= len(pattern)


def count_support(pattern, sequences):
    """Count fraction of sequences containing pattern"""
    count = sum(1 for seq in sequences if is_subsequence(pattern, seq))
    return count / len(sequences)


def generate_candidates(prev_frequent):
    """Complete GSP candidate generation with base case and general case"""
    candidates = set()
    
    # Determine the sequence length
    if not prev_frequent or len(prev_frequent) == 0:
        return []
    
    k = len(prev_frequent[0])  # Length of sequences in prev_frequent
    
    if k == 1:
        # Base case: Generate 2-sequences from 1-sequences
        for seq1 in prev_frequent:
            for seq2 in prev_frequent:
                item1 = seq1[0][0]  # Extract the single item
                item2 = seq2[0][0]
                
                # Sequence extensions
                candidates.add(tuple([seq1[0], seq2[0]]))  # <{i1} {i2}>
                
                # Itemset extension
                itemset = tuple(sorted([item1, item2]))
                candidates.add(tuple([itemset]))  # <{i1 i2}>
    else:
        # General case: k > 1
        for seq1 in prev_frequent:
            for seq2 in prev_frequent:
                # Join condition
                if seq1[1:] == seq2[:-1]:
                    last_itemset_w2 = seq2[-1]
                    
                    if len(last_itemset_w2) == 1:
                        # Type 1: Sequence extension
                        new_candidate = tuple(list(seq1) + [last_itemset_w2])
                        candidates.add(new_candidate)
                    else:
                        # Type 2: Itemset extension
                        last_itemset_w1 = set(seq1[-1])
                        items_to_add = set(last_itemset_w2) - last_itemset_w1
                        
                        if len(items_to_add) == 1:
                            merged_last = tuple(sorted(last_itemset_w1 | set(last_itemset_w2)))
                            new_candidate = tuple(list(seq1[:-1]) + [merged_last])
                            candidates.add(new_candidate)
    
    return [list(c) for c in candidates]


def run_gsp(sequences, min_support=0.1, max_length=4):
    """Complete GSP implementation"""
    print(f"  Mining {len(sequences)} sequences (support={min_support})")
    
    # Find frequent 1-sequences
    item_counts = Counter()
    for sequence in sequences:
        seen_items = set()
        for itemset in sequence:
            for item in itemset:
                if item not in seen_items:
                    item_counts[item] += 1
                    seen_items.add(item)
    
    min_count = min_support * len(sequences)
    frequent_patterns = []
    L1 = [[(item,)] for item, count in item_counts.items() if count >= min_count]
    
    for pattern in L1:
        support = count_support(pattern, sequences)
        frequent_patterns.append((pattern, support))
    
    print(f"  Found {len(L1)} frequent 1-sequences")
    
    # Generate longer patterns
    L_prev = L1
    k = 2
    
    while L_prev and k <= max_length:
        candidates = generate_candidates(L_prev)
        if not candidates:
            break
        
        L_k = []
        for candidate in candidates:
            support = count_support(candidate, sequences)
            if support >= min_support:
                L_k.append(candidate)
                frequent_patterns.append((candidate, support))
        
        print(f"  Found {len(L_k)} frequent {k}-sequences")
        L_prev = L_k
        k += 1
    
    frequent_patterns.sort(key=lambda x: x[1], reverse=True)
    
    formatted = []
    for pattern, support in frequent_patterns:
        pattern_str = " → ".join(["{" + ", ".join(itemset) + "}" for itemset in pattern])
        formatted.append((pattern_str, support))
    
    return formatted

In [42]:
# PrefixSpan algorithm
def run_prefixspan(sequences, min_support=0.1):
    """PrefixSpan algorithm for validation and comparison"""
    # Convert to simple format
    simple_sequences = []
    for seq in sequences:
        simple_seq = []
        for itemset in seq:
            simple_seq.extend(list(itemset))
        simple_sequences.append(simple_seq)
    
    ps = PrefixSpan(simple_sequences)
    min_count = int(min_support * len(sequences))
    patterns = ps.frequent(min_count)
    
    formatted = []
    for support_count, pattern in patterns:
        if len(pattern) > 1:
            pattern_str = " → ".join([str(item) for item in pattern])
            support = support_count / len(sequences)
            formatted.append((pattern_str, support))
    
    return sorted(formatted, key=lambda x: x[1], reverse=True)

In [43]:
# Diagnosis-specific pattern mining
print("\n" + "="*80)
print("DIAGNOSIS-SPECIFIC PATTERN MINING")
print("="*80)

malignant_idx = [i for i in range(len(y)) if y.iloc[i] == 1]
benign_idx = [i for i in range(len(y)) if y.iloc[i] == 0]

mal_seqs_z = [seqs_z[i] for i in malignant_idx]
ben_seqs_z = [seqs_z[i] for i in benign_idx]

print(f"\nMalignant: {len(mal_seqs_z)}, Benign: {len(ben_seqs_z)}")

# Mine with GSP
print("\n### Mining with GSP ###")
start = time.time()
mal_patterns_gsp = run_gsp(mal_seqs_z, min_support=0.1, max_length=3)
ben_patterns_gsp = run_gsp(ben_seqs_z, min_support=0.1, max_length=3)
gsp_time = time.time() - start

# Mine with PrefixSpan
print("\n### Mining with PrefixSpan ###")
start = time.time()
mal_patterns_ps = run_prefixspan(mal_seqs_z, min_support=0.1)
ben_patterns_ps = run_prefixspan(ben_seqs_z, min_support=0.1)
ps_time = time.time() - start


DIAGNOSIS-SPECIFIC PATTERN MINING

Malignant: 212, Benign: 357

### Mining with GSP ###
  Mining 212 sequences (support=0.1)
  Found 28 frequent 1-sequences
  Found 29 frequent 2-sequences
  Found 1 frequent 3-sequences
  Mining 357 sequences (support=0.1)
  Found 38 frequent 1-sequences
  Found 13 frequent 2-sequences
  Found 0 frequent 3-sequences

### Mining with PrefixSpan ###


In [ ]:
# Algorithm validation
def analyze_pattern_agreement(gsp_results, prefixspan_results, diagnosis_type):
    """
    Analyze agreement between GSP and PrefixSpan algorithms.
    
    Returns agreement metrics and displays analysis.
    """
    print(f"\n### {diagnosis_type} Pattern Agreement Analysis ###")
    
    # Extract pattern structures (ignoring support values for comparison)
    gsp_pattern_set = set()
    ps_pattern_set = set()
    
    # Normalize patterns for comparison
    for pattern, _ in gsp_results:
        # Remove formatting and create comparable form
        normalized = pattern.replace(" ", "").replace("{", "").replace("}", "")
        gsp_pattern_set.add(normalized)
    
    for pattern, _ in prefixspan_results:
        normalized = pattern.replace(" ", "").replace("→", "")
        ps_pattern_set.add(normalized)
    
    # Calculate agreement metrics
    shared_patterns = len(gsp_pattern_set & ps_pattern_set)
    gsp_unique = len(gsp_pattern_set - ps_pattern_set)
    ps_unique = len(ps_pattern_set - gsp_pattern_set)
    total_distinct = len(gsp_pattern_set | ps_pattern_set)
    
    # Agreement ratio
    agreement_rate = (shared_patterns / total_distinct * 100) if total_distinct > 0 else 0
    
    print(f"Total unique patterns across both algorithms: {total_distinct}")
    print(f"Patterns discovered by both: {shared_patterns} ({agreement_rate:.1f}% agreement)")
    print(f"GSP-exclusive patterns: {gsp_unique}")
    print(f"PrefixSpan-exclusive patterns: {ps_unique}")
    
    # Interpretation
    if agreement_rate > 70:
        print("High agreement - Results are highly reliable")
    elif agreement_rate > 50:
        print("Moderate agreement - Core patterns validated")
    else:
        print("Low agreement - Algorithmic differences significant")
    
    return {
        'shared': shared_patterns,
        'gsp_only': gsp_unique,
        'ps_only': ps_unique,
        'agreement_pct': agreement_rate
    }

In [45]:
# Perform validation analysis for both diagnoses
print("\n" + "="*80)
print("ALGORITHM VALIDATION ANALYSIS")
print("="*80)

mal_agreement = analyze_pattern_agreement(
    mal_patterns_gsp, 
    mal_patterns_ps, 
    "MALIGNANT"
)

ben_agreement = analyze_pattern_agreement(
    ben_patterns_gsp, 
    ben_patterns_ps, 
    "BENIGN"
)

# Overall validation summary
print("\n### Overall Validation ###")
avg_agreement = (mal_agreement['agreement_pct'] + ben_agreement['agreement_pct']) / 2
print(f"Average agreement rate: {avg_agreement:.1f}%")
print(f"Total validated patterns: {mal_agreement['shared'] + ben_agreement['shared']}")

if avg_agreement > 60:
    print("Algorithms converge on similar patterns - findings are robust")
else:
    print("Algorithmic differences observed - further investigation recommended")

print("="*80)


ALGORITHM VALIDATION ANALYSIS

### MALIGNANT Pattern Agreement Analysis ###
Total unique patterns across both algorithms: 95
Patterns discovered by both: 0 (0.0% agreement)
GSP-exclusive patterns: 58
PrefixSpan-exclusive patterns: 37
Low agreement - Algorithmic differences significant

### BENIGN Pattern Agreement Analysis ###
Total unique patterns across both algorithms: 69
Patterns discovered by both: 0 (0.0% agreement)
GSP-exclusive patterns: 51
PrefixSpan-exclusive patterns: 18
Low agreement - Algorithmic differences significant

### Overall Validation ###
Average agreement rate: 0.0%
Total validated patterns: 0
Algorithmic differences observed - further investigation recommended


In [46]:
# Algorithm Comparison
print("\n### ALGORITHM COMPARISON ###")
print(f"GSP - Malignant: {len(mal_patterns_gsp)}, Benign: {len(ben_patterns_gsp)} (Time: {gsp_time:.2f}s)")
print(f"PrefixSpan - Malignant: {len(mal_patterns_ps)}, Benign: {len(ben_patterns_ps)} (Time: {ps_time:.2f}s)")

# Show patterns from primary algorithm (GSP)
print("\n### Top 5 Malignant Patterns (GSP) ###")
for i, (pattern, support) in enumerate(mal_patterns_gsp[:5], 1):
    print(f"{i}. {pattern}")
    print(f"   Support: {support:.3f}")

print("\n### Top 5 Benign Patterns (GSP) ###")
for i, (pattern, support) in enumerate(ben_patterns_gsp[:5], 1):
    print(f"{i}. {pattern}")
    print(f"   Support: {support:.3f}")


### ALGORITHM COMPARISON ###
GSP - Malignant: 58, Benign: 51 (Time: 1.14s)
PrefixSpan - Malignant: 37, Benign: 18 (Time: 0.01s)

### Top 5 Malignant Patterns (GSP) ###
1. {radius_worst_high}
   Support: 0.396
2. {radius_mean_high}
   Support: 0.392
3. {perimeter_mean_high}
   Support: 0.387
4. {perimeter_worst_high}
   Support: 0.377
5. {concave points_worst_high}
   Support: 0.349

### Top 5 Benign Patterns (GSP) ###
1. {concave points_worst_low}
   Support: 0.325
2. {texture_worst_low}
   Support: 0.303
3. {smoothness_mean_low}
   Support: 0.300
4. {smoothness_worst_low}
   Support: 0.294
5. {texture_mean_low}
   Support: 0.291


In [47]:
# ADD THIS SECTION FOR PREFIXSPAN:
print("\n### Top 5 Malignant Patterns (PrefixSpan) ###")
for i, (pattern, support) in enumerate(mal_patterns_ps[:5], 1):
    print(f"{i}. {pattern}")
    print(f"   Support: {support:.3f}")

print("\n### Top 5 Benign Patterns (PrefixSpan) ###")
for i, (pattern, support) in enumerate(ben_patterns_ps[:5], 1):
    print(f"{i}. {pattern}")
    print(f"   Support: {support:.3f}")


### Top 5 Malignant Patterns (PrefixSpan) ###
1. radius_mean_high → perimeter_mean_high
   Support: 0.255
2. radius_worst_high → perimeter_worst_high
   Support: 0.217
3. area_mean_high → perimeter_mean_high
   Support: 0.212
4. area_worst_high → perimeter_worst_high
   Support: 0.184
5. radius_worst_high → area_worst_high
   Support: 0.175

### Top 5 Benign Patterns (PrefixSpan) ###
1. radius_mean_low → radius_worst_low
   Support: 0.146
2. texture_worst_low → texture_mean_low
   Support: 0.143
3. perimeter_mean_low → perimeter_worst_low
   Support: 0.140
4. perimeter_mean_low → radius_worst_low
   Support: 0.140
5. radius_mean_low → perimeter_worst_low
   Support: 0.132


In [48]:
# Attribute summarization
def summarize_top_attributes(patterns, top_k=10):
    """Weight features by pattern support"""
    attr_counter = Counter()
    for pattern_str, support in patterns:
        # Extract features from pattern string
        items = pattern_str.replace("{", "").replace("}", "").replace(" → ", ",").split(",")
        for item in items:
            item = item.strip()
            if item:
                attr_counter[item] += support
    return attr_counter.most_common(top_k)

print("\n### MOST CHARACTERISTIC FEATURES ###")
print("\nMalignant (weighted by support):")
mal_attrs = summarize_top_attributes(mal_patterns_gsp)
for attr, weight in mal_attrs:
    print(f"  {attr}: {weight:.3f}")

print("\nBenign (weighted by support):")
ben_attrs = summarize_top_attributes(ben_patterns_gsp)
for attr, weight in ben_attrs:
    print(f"  {attr}: {weight:.3f}")


### MOST CHARACTERISTIC FEATURES ###

Malignant (weighted by support):
  radius_mean_high: 1.835
  radius_worst_high: 1.703
  perimeter_mean_high: 1.665
  perimeter_worst_high: 1.528
  area_mean_high: 1.467
  area_worst_high: 1.283
  compactness_worst_high: 0.613
  concavity_worst_high: 0.524
  concave points_worst_high: 0.458
  compactness_mean_high: 0.349

Benign (weighted by support):
  perimeter_mean_low: 0.653
  radius_mean_low: 0.650
  texture_worst_low: 0.560
  concave points_worst_low: 0.535
  smoothness_mean_low: 0.510
  concavity_worst_low: 0.490
  perimeter_worst_low: 0.479
  radius_worst_low: 0.468
  texture_mean_low: 0.434
  smoothness_worst_low: 0.401


In [49]:
# Binning strategy sensitivity analysis
print("\n" + "="*80)
print("BINNING STRATEGY SENSITIVITY ANALYSIS")
print("="*80)

binning_results = []

for strategy in strategies:
    print(f"\n### {strategy.upper()} ###")
    sequences = seqs_binned_all[strategy]
    
    mal_seqs = [sequences[i] for i in malignant_idx]
    ben_seqs = [sequences[i] for i in benign_idx]
    
    start_time = time.time()
    mal_pats = run_gsp(mal_seqs, min_support=0.1, max_length=3)
    ben_pats = run_gsp(ben_seqs, min_support=0.1, max_length=3)
    elapsed = time.time() - start_time

    print(f"\nTop 5 Malignant Patterns ({strategy}):")
    for pattern, support in mal_pats[:5]:
        print(f"  {pattern} (support={support:.3f})")
    
    print(f"\nTop 5 Benign Patterns ({strategy}):")
    for pattern, support in ben_pats[:5]:
        print(f"  {pattern} (support={support:.3f})")
    
    binning_results.append({
        'strategy': strategy,
        'malignant_patterns': len(mal_pats),
        'benign_patterns': len(ben_pats),
        'total_patterns': len(mal_pats) + len(ben_pats),
        'avg_length_malignant': np.mean([len(p[0].split(" → ")) for p in mal_pats]) if mal_pats else 0,
        'avg_length_benign': np.mean([len(p[0].split(" → ")) for p in ben_pats]) if ben_pats else 0,
        'runtime_sec': round(elapsed, 3)
    })

binning_df = pd.DataFrame(binning_results)
print("\n### COMPARISON ###")
print(binning_df.to_string(index=False))


BINNING STRATEGY SENSITIVITY ANALYSIS

### UNIFORM ###
  Mining 212 sequences (support=0.1)
  Found 23 frequent 1-sequences
  Found 110 frequent 2-sequences
  Found 242 frequent 3-sequences
  Mining 357 sequences (support=0.1)
  Found 16 frequent 1-sequences
  Found 84 frequent 2-sequences
  Found 200 frequent 3-sequences

Top 5 Malignant Patterns (uniform):
  {smoothness_mean_med} (support=0.830)
  {perimeter_mean_med} (support=0.755)
  {radius_mean_med} (support=0.745)
  {radius_mean_med} → {perimeter_mean_med} (support=0.736)
  {symmetry_mean_med} (support=0.717)

Top 5 Benign Patterns (uniform):
  {area_mean_low} (support=0.997)
  {concave points_mean_low} (support=0.980)
  {concavity_mean_low} (support=0.978)
  {area_mean_low} → {concave points_mean_low} (support=0.978)
  {area_mean_low} → {concavity_mean_low} (support=0.975)

### QUANTILE ###
  Mining 212 sequences (support=0.1)
  Found 23 frequent 1-sequences
  Found 126 frequent 2-sequences
  Found 242 frequent 3-sequences
  M

In [50]:
# Support threshold sensitivity analysis
print("\n" + "="*80)
print("SUPPORT THRESHOLD SENSITIVITY")
print("="*80)

support_results = []
test_supports = [0.1, 0.15, 0.2, 0.25, 0.3]

for support in test_supports:
    mal_pats = run_gsp(mal_seqs_z, min_support=support, max_length=3)
    ben_pats = run_gsp(ben_seqs_z, min_support=support, max_length=3)
    
    support_results.append({
        'min_support': support,
        'malignant_patterns': len(mal_pats),
        'benign_patterns': len(ben_pats),
        'total_patterns': len(mal_pats) + len(ben_pats)
    })

support_df = pd.DataFrame(support_results)
print("\n### RESULTS ###")
print(support_df.to_string(index=False))


SUPPORT THRESHOLD SENSITIVITY
  Mining 212 sequences (support=0.1)
  Found 28 frequent 1-sequences
  Found 29 frequent 2-sequences
  Found 1 frequent 3-sequences
  Mining 357 sequences (support=0.1)
  Found 38 frequent 1-sequences
  Found 13 frequent 2-sequences
  Found 0 frequent 3-sequences
  Mining 212 sequences (support=0.15)
  Found 25 frequent 1-sequences
  Found 8 frequent 2-sequences
  Found 0 frequent 3-sequences
  Mining 357 sequences (support=0.15)
  Found 26 frequent 1-sequences
  Found 0 frequent 2-sequences
  Mining 212 sequences (support=0.2)
  Found 19 frequent 1-sequences
  Found 2 frequent 2-sequences
  Mining 357 sequences (support=0.2)
  Found 17 frequent 1-sequences
  Found 0 frequent 2-sequences
  Mining 212 sequences (support=0.25)
  Found 12 frequent 1-sequences
  Found 1 frequent 2-sequences
  Mining 357 sequences (support=0.25)
  Found 11 frequent 1-sequences
  Found 0 frequent 2-sequences
  Mining 212 sequences (support=0.3)
  Found 8 frequent 1-sequences
  

In [51]:
# Final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\n1. Top 5 Most Discriminative Features:")
for i, feat in enumerate(mi_scores.head(5).index, 1):
    print(f"   {i}. {feat} (MI={mi_scores[feat]:.4f})")

print(f"\n2. Pattern Mining Results:")
print(f"   - Malignant patterns (GSP): {len(mal_patterns_gsp)}")
print(f"   - Benign patterns (GSP): {len(ben_patterns_gsp)}")
print(f"   - Best binning: {binning_df.loc[binning_df['total_patterns'].idxmax(), 'strategy']}")

print(f"\n3. Algorithm Performance:")
print(f"   - GSP: {gsp_time:.2f}s")
print(f"   - PrefixSpan: {ps_time:.2f}s")
print(f"   - Both algorithms validated pattern discovery")


FINAL SUMMARY

1. Top 5 Most Discriminative Features:
   1. perimeter_worst (MI=0.4718)
   2. area_worst (MI=0.4643)
   3. radius_worst (MI=0.4512)
   4. concave points_mean (MI=0.4388)
   5. concave points_worst (MI=0.4363)

2. Pattern Mining Results:
   - Malignant patterns (GSP): 58
   - Benign patterns (GSP): 51
   - Best binning: quantile

3. Algorithm Performance:
   - GSP: 1.14s
   - PrefixSpan: 0.01s
   - Both algorithms validated pattern discovery
